In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.window import Window

In [0]:
silver_stock_prices = spark.read.table("raj.silver.stock_prices")
dim_date = spark.read.table("raj.gold.dim_date")
dim_company = spark.read.table("raj.gold.dim_company")

fact_df = silver_stock_prices.join(dim_date,silver_stock_prices.trade_date==dim_date.full_date,"inner")\
    .join(dim_company,(silver_stock_prices.ticker==dim_company.ticker) & (dim_company.is_current==True),"inner")
silver_stock_prices.printSchema()
fact_df.printSchema()

In [0]:
final_fact_df = fact_df.select(
    dim_date.date_key.alias("date_key"),
    dim_company.company_key.alias("company_key"),

    silver_stock_prices.ticker.alias("ticker"),
    silver_stock_prices.trade_date.alias("trade_date"),

    silver_stock_prices.close_price,
    silver_stock_prices.high_price,
    silver_stock_prices.low_price,
    silver_stock_prices.open_price,
    silver_stock_prices.volume,
    silver_stock_prices.daily_range,
    silver_stock_prices.daily_return,
    silver_stock_prices.moving_avg_7d,
    silver_stock_prices.moving_avg_30d
)

In [0]:
final_fact_df.select("ticker").distinct().count()

In [0]:
final_fact_df.write.format("delta").mode("overwrite").option("overwriteSchema","True").saveAsTable("raj.gold.fact_stock_prices")

In [0]:
%sql
SELECT 
    c.sector,
    COUNT(*) as trading_days,
    ROUND(AVG(f.close_price), 2) as avg_price,
    ROUND(AVG(f.daily_return), 2) as avg_return,
    ROUND(AVG(f.volume), 0) as avg_volume
FROM raj.gold.fact_stock_prices f
JOIN raj.gold.dim_company c ON f.company_key = c.company_key
JOIN raj.gold.dim_date d ON f.date_key = d.date_key
GROUP BY c.sector
ORDER BY avg_return DESC